In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, re
import glob

In [2]:
# ── Standardize team names in ALL games_live2 CSV files ──
# Covers filenames AND content columns (away_team_abbrev, home_team_abbrev)

HIST_TO_MODERN = {
    # Relocations / renames
    "SEA": "OKC",   # SuperSonics -> Thunder
    "VAN": "MEM",   # Vancouver Grizzlies -> Memphis
    "NJN": "BKN",   # New Jersey Nets -> Brooklyn
    "NJ":  "BKN",   # New Jersey Nets (short form)
    "BRK": "BKN",   # Brooklyn alt code
    "NOH": "NOP",   # New Orleans Hornets -> Pelicans
    "NOK": "NOP",   # New Orleans/Oklahoma City Hornets -> Pelicans
    "NO":  "NOP",   # New Orleans (short form)

    # Charlotte franchise codes across eras
    "CHH": "CHA",
    "CHO": "CHA",
    "CHA": "CHA",

    # Washington historical codes
    "WSB": "WAS",
    "WSH": "WAS",   # common ESPN-style code
    "BAL": "WAS",
    "WAS": "WAS",

    # Common abbreviation variants (ESPN / thesports style)
    "PHO": "PHX",
    "GS":  "GSW",
    "SA":  "SAS",
    "NY":  "NYK",
    "BK":  "BKN",
    "UTAH": "UTA",

    # Modern 30 teams (identity mappings for completeness)
    "ATL": "ATL", "BOS": "BOS", "BKN": "BKN", "CHI": "CHI", "CLE": "CLE",
    "DAL": "DAL", "DEN": "DEN", "DET": "DET", "GSW": "GSW", "HOU": "HOU",
    "IND": "IND", "LAC": "LAC", "LAL": "LAL", "MEM": "MEM", "MIA": "MIA",
    "MIL": "MIL", "MIN": "MIN", "NOP": "NOP", "NYK": "NYK", "OKC": "OKC",
    "ORL": "ORL", "PHI": "PHI", "PHX": "PHX", "POR": "POR", "SAC": "SAC",
    "SAS": "SAS", "TOR": "TOR", "UTA": "UTA",
}

# All-Star / special event codes to skip (leave as-is)
SPECIAL_TEAMS = {"EAST", "WEST", "USA", "WORLD", "CAN", "DUR", "GIA",
                 "KEN", "LEB", "CHK", "SHQ", "STE"}

def canonical(abbrev):
    """Map any team abbreviation to its modern canonical form."""
    if abbrev is None or (isinstance(abbrev, float) and np.isnan(abbrev)):
        return abbrev
    a = str(abbrev).strip().upper()
    if a in SPECIAL_TEAMS:
        return a  # leave All-Star teams untouched
    return HIST_TO_MODERN.get(a, a)

# Team-related content columns to standardize inside each CSV
TEAM_COLS = ["away_team_abbrev", "home_team_abbrev",
             "away_team_name_alt", "home_team_name_alt"]

In [4]:
# Load all schedule files to create game_id -> game_date mapping
schedule_files = glob.glob("data/schedules/schedule_*.csv")
print(f"Loading {len(schedule_files)} schedule files...")

game_date_map = {}
for sched_file in schedule_files:
    # Schedules from hoopR use lowercase (game_id, game_date); legacy NBA Stats
    # API schedules use uppercase (GAME_ID, GAME_DATE). Handle both.
    df = pd.read_csv(sched_file, dtype={"GAME_ID": str, "game_id": str})
    gid_col = "GAME_ID" if "GAME_ID" in df.columns else "game_id"
    gdate_col = "GAME_DATE" if "GAME_DATE" in df.columns else "game_date"

    for _, row in df.iterrows():
        game_id = str(row[gid_col]).zfill(10)
        game_date_map[game_id] = row[gdate_col]

print(f"Loaded {len(game_date_map)} games with dates")
print(f"Sample: {list(game_date_map.items())[:3]}")


Loading 16 schedule files...
Loaded 20478 games with dates
Sample: [('0401344140', '2021-07-20'), ('0401344139', '2021-07-17'), ('0401344138', '2021-07-14')]


In [5]:
# Load games.csv
team_stats = pd.read_csv("data/team_stats.csv", dtype={"game_id": str})
print(f"Loaded {len(team_stats)} games from team_stats.csv")

Loaded 20327 games from team_stats.csv


In [6]:
# filter out games where away_team is not in the hist to moderm dict
print("before filtering", len(team_stats))
team_stats = team_stats[team_stats['away_team'].isin(HIST_TO_MODERN.keys())]
print("after filtering", len(team_stats))

before filtering 20327
after filtering 20305


In [7]:
# Calculate running win/loss records for each team within each season
# + days rest (days since last game, capped at 7)
# For each game, look at all PREVIOUS games in the same season to compute records
# (no future leakage, no cross-season bleed)

print("Computing running win/loss records per season + days rest...")

# Ensure sorted by date
team_stats['game_date'] = pd.to_datetime(team_stats['game_date'])
team_stats = team_stats.sort_values('game_date').reset_index(drop=True)

# Initialize new columns
team_stats['home_wins'] = 0
team_stats['home_losses'] = 0
team_stats['away_wins'] = 0
team_stats['away_losses'] = 0
team_stats['home_days_rest'] = 7
team_stats['away_days_rest'] = 7

# Track records per season per team: {season: {team: {'wins': int, 'losses': int}}}
season_records = {}

# Track last game date per team (NOT per season — rest spans offseason too)
team_last_game_date = {}

MAX_REST = 7

for idx, row in team_stats.iterrows():
    season = row['season']
    home_team = row['home_team']
    away_team = row['away_team']
    game_date = row['game_date']
    
    # Initialize season tracking if needed
    if season not in season_records:
        season_records[season] = {}
    
    if home_team not in season_records[season]:
        season_records[season][home_team] = {'wins': 0, 'losses': 0}
    
    if away_team not in season_records[season]:
        season_records[season][away_team] = {'wins': 0, 'losses': 0}
    
    # Set the current record BEFORE this game (no leakage)
    team_stats.at[idx, 'home_wins'] = season_records[season][home_team]['wins']
    team_stats.at[idx, 'home_losses'] = season_records[season][home_team]['losses']
    team_stats.at[idx, 'away_wins'] = season_records[season][away_team]['wins']
    team_stats.at[idx, 'away_losses'] = season_records[season][away_team]['losses']
    
    # Compute days rest (days since last game, capped at MAX_REST)
    if home_team in team_last_game_date:
        home_rest = (game_date - team_last_game_date[home_team]).days
        team_stats.at[idx, 'home_days_rest'] = min(home_rest, MAX_REST)
    # else: stays at default 7
    
    if away_team in team_last_game_date:
        away_rest = (game_date - team_last_game_date[away_team]).days
        team_stats.at[idx, 'away_days_rest'] = min(away_rest, MAX_REST)
    # else: stays at default 7
    
    # Determine who won this game by comparing points, then update records
    home_pts = row['home_PTS']
    away_pts = row['away_PTS']
    
    if pd.notna(home_pts) and pd.notna(away_pts):
        if float(home_pts) > float(away_pts):
            # Home team won
            season_records[season][home_team]['wins'] += 1
            season_records[season][away_team]['losses'] += 1
        elif float(away_pts) > float(home_pts):
            # Away team won
            season_records[season][away_team]['wins'] += 1
            season_records[season][home_team]['losses'] += 1
    
    # Update last game date for both teams
    team_last_game_date[home_team] = game_date
    team_last_game_date[away_team] = game_date
    
    if (idx + 1) % 5000 == 0:
        print(f"Processed {idx + 1}/{len(team_stats)} games...")

print(f"\nDone! Added columns: home_wins, home_losses, away_wins, away_losses, home_days_rest, away_days_rest")
print(f"Seasons found: {sorted(season_records.keys())}")

# Show sample of mid-season games so records are non-zero
mid_season = team_stats[(team_stats['home_wins'] + team_stats['home_losses'] > 10)].head(10)
print(f"\nSample records (mid-season games):")
print(mid_season[['game_date', 'season', 'home_team', 'away_team', 'home_PTS', 'away_PTS', 
                   'home_wins', 'home_losses', 'away_wins', 'away_losses',
                   'home_days_rest', 'away_days_rest']].to_string(index=False))

# Show rest distribution
print(f"\nDays rest distribution (home):")
print(team_stats['home_days_rest'].value_counts().sort_index())
print(f"\nDays rest distribution (away):")
print(team_stats['away_days_rest'].value_counts().sort_index())

Computing running win/loss records per season + days rest...
Processed 5000/20305 games...
Processed 10000/20305 games...
Processed 15000/20305 games...
Processed 20000/20305 games...

Done! Added columns: home_wins, home_losses, away_wins, away_losses, home_days_rest, away_days_rest
Seasons found: ['2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']

Sample records (mid-season games):
 game_date  season home_team away_team  home_PTS  away_PTS  home_wins  home_losses  away_wins  away_losses  home_days_rest  away_days_rest
2010-11-16 2010-11       MEM       POR        99       100          4            7          6            5               1               3
2010-11-17 2010-11       PHI       TOR        86        94          2            9          2            9               1               1
2010-11-17 2010-11       DET       LAL        90       103          4 

In [8]:
# Save the dataframe with win/loss records back to training_games.csv
output_file = "data/training_games.csv"
team_stats.to_csv(output_file, index=False)
print(f"✓ Saved {len(team_stats)} games with win/loss records to {output_file}")
print(f"New columns added: home_wins, home_losses, away_wins, away_losses")

✓ Saved 20305 games with win/loss records to data/training_games.csv
New columns added: home_wins, home_losses, away_wins, away_losses


In [9]:
training_games2 = pd.read_csv('data/training_games.csv')

In [10]:
def convert_minutes_to_decimal(minutes_str):
    """Convert MM:SS format to decimal minutes."""
    if pd.isna(minutes_str):
        return 0.0
    
    minutes_str = str(minutes_str).strip()
    
    # If already a number, return it
    try:
        return float(minutes_str)
    except ValueError:
        pass
    
    # Parse MM:SS format
    if ':' in minutes_str:
        parts = minutes_str.split(':')
        if len(parts) == 2:
            try:
                mins = int(parts[0])
                secs = int(parts[1])
                return float(mins + (secs / 60.0))
            except ValueError:
                return 0.0
    
    return 0.0

# Test the function
print(f"Test: '22:07' -> {convert_minutes_to_decimal('22:07'):.2f} minutes")
print(f"Test: '15:30' -> {convert_minutes_to_decimal('15:30'):.2f} minutes")
print(f"Test: '3:45' -> {convert_minutes_to_decimal('3:45'):.2f} minutes")
print(f"Test: 25.5 -> {convert_minutes_to_decimal(25.5):.2f} minutes")

Test: '22:07' -> 22.12 minutes
Test: '15:30' -> 15.50 minutes
Test: '3:45' -> 3.75 minutes
Test: 25.5 -> 25.50 minutes


In [11]:
# Get all player game files to create a mapping from game_id to file
# game_rosters is nested by season year: data/game_rosters/2009/281028002_CLE_BOS.csv
player_game_files = glob.glob("data/game_rosters/*/*.csv")
game_id_to_file = {}

for game_file in player_game_files:
    filename = os.path.basename(game_file)
    game_id = filename.split("_")[0]  # keep raw game_id without zero-padding
    game_id_to_file[game_id] = game_file

print(f"Found {len(game_id_to_file)} player game files")
print(f"Sample game_ids: {list(game_id_to_file.keys())[:5]}")

Found 20377 player game files
Sample game_ids: ['400277794', '400277813', '400278020', '400278868', '400277989']


In [12]:
# Define specific stats to compute recency-weighted averages for
stat_names = ['FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 
              'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS']

# Recency weighting config
DECAY = 0.9       # Exponential decay factor (0.9^t, t=1 most recent -> highest weight)
MIN_GAMES = 4     # Minimum games played in current season before including in dataset
TEAM_ROLLING_WINDOW = 10  # Rolling window for recent-form team stats

# Ensure chronological order (required: history must only include past games, no leakage)
training_games2['game_date'] = pd.to_datetime(training_games2['game_date'])
training_games2 = training_games2.sort_values('game_date').reset_index(drop=True)

# Track each team's game history PER SEASON (no cross-season bleed)
# Key: (team_abbr, season) -> list of stat dicts in chronological order
team_season_history = {}

# Store rows that have sufficient season history
training_rows = []
skipped_insufficient_history = 0

def compute_weighted_stats(history, stat_names, decay):
    """
    Compute recency-weighted averages for a team's season history.
    history: list of stat dicts in chronological order (oldest first)
    Returns dict of {stat_name: weighted_average}
    
    Weighting: most recent game gets 0.9^1, second most recent gets 0.9^2, etc.
    All weights are normalized to sum to 1.
    """
    n = len(history)
    # Reverse so index 0 = most recent game
    games_reversed = list(reversed(history))
    
    # Raw weights: 0.9^1, 0.9^2, ..., 0.9^n
    raw_weights = [decay ** (t + 1) for t in range(n)]
    weight_total = sum(raw_weights)
    norm_weights = [w / weight_total for w in raw_weights]
    
    result = {}
    for stat in stat_names:
        weighted_sum = 0.0
        weight_used = 0.0
        for i, game in enumerate(games_reversed):
            val = game.get(stat)
            if val is not None and not np.isnan(val):
                weighted_sum += norm_weights[i] * val
                weight_used += norm_weights[i]
        
        if weight_used > 0:
            # Re-normalize in case some games had NaN for this stat
            result[stat] = weighted_sum / weight_used
        else:
            result[stat] = np.nan
    
    return result

def compute_rolling_stats(history, stat_names, window):
    """
    Simple rolling average of the last `window` games.
    history: list of stat dicts in chronological order (oldest first).
    Returns dict of {stat_name: mean} using only the last `window` games.
    """
    recent = history[-window:]  # last N games
    result = {}
    for stat in stat_names:
        vals = [g.get(stat) for g in recent if g.get(stat) is not None and not np.isnan(g.get(stat))]
        result[stat] = np.mean(vals) if vals else np.nan
    return result

for idx, row in training_games2.iterrows():
    away_team = row['away_team']
    home_team = row['home_team']
    season = row['season']
    
    away_key = (away_team, season)
    home_key = (home_team, season)
    
    # Get season-specific history (only games from this season)
    away_history = team_season_history.get(away_key, [])
    home_history = team_season_history.get(home_key, [])
    
    if len(away_history) < MIN_GAMES or len(home_history) < MIN_GAMES:
        # Not enough season history yet, skip this game
        skipped_insufficient_history += 1
    else:
        # Start with all original data
        new_row = row.to_dict()
        
        # Store original PTS values before we replace them
        original_away_PTS = new_row.get('away_PTS')
        original_home_PTS = new_row.get('home_PTS')
        
        # Compute recency-weighted stats for away team (season only)
        away_weighted = compute_weighted_stats(away_history, stat_names, DECAY)
        for stat in stat_names:
            new_row[f'away_{stat}'] = away_weighted[stat]
        
        # Compute recency-weighted stats for home team (season only)
        home_weighted = compute_weighted_stats(home_history, stat_names, DECAY)
        for stat in stat_names:
            new_row[f'home_{stat}'] = home_weighted[stat]
        
        # Compute 10-game rolling averages for recent form
        if len(away_history) >= TEAM_ROLLING_WINDOW:
            away_rolling = compute_rolling_stats(away_history, stat_names, TEAM_ROLLING_WINDOW)
            for stat in stat_names:
                new_row[f'away_r10_{stat}'] = away_rolling[stat]
        else:
            for stat in stat_names:
                new_row[f'away_r10_{stat}'] = np.nan
        
        if len(home_history) >= TEAM_ROLLING_WINDOW:
            home_rolling = compute_rolling_stats(home_history, stat_names, TEAM_ROLLING_WINDOW)
            for stat in stat_names:
                new_row[f'home_r10_{stat}'] = home_rolling[stat]
        else:
            for stat in stat_names:
                new_row[f'home_r10_{stat}'] = np.nan
        
        # Add back the original PTS columns
        new_row['away_PTS_actual'] = original_away_PTS
        new_row['home_PTS_actual'] = original_home_PTS
        
        training_rows.append(new_row)
    
    # Update team's season history with current game stats (for future games)
    # Away team stats
    if away_key not in team_season_history:
        team_season_history[away_key] = []
    
    away_game_stats = {}
    for stat in stat_names:
        away_col = f'away_{stat}'
        if away_col in row.index:
            val = row[away_col]
            try:
                away_game_stats[stat] = float(val) if not pd.isna(val) else np.nan
            except (ValueError, TypeError):
                away_game_stats[stat] = np.nan
    team_season_history[away_key].append(away_game_stats)
    
    # Home team stats
    if home_key not in team_season_history:
        team_season_history[home_key] = []
    
    home_game_stats = {}
    for stat in stat_names:
        home_col = f'home_{stat}'
        if home_col in row.index:
            val = row[home_col]
            try:
                home_game_stats[stat] = float(val) if not pd.isna(val) else np.nan
            except (ValueError, TypeError):
                home_game_stats[stat] = np.nan
    team_season_history[home_key].append(home_game_stats)
    
    # Progress indicator
    if (idx + 1) % 5000 == 0:
        print(f"Processed {idx + 1}/{len(training_games2)} games, {len(training_rows)} included...")

print(f"\n{'='*60}")
print(f"Processing complete:")
print(f"  Total games processed: {len(training_games2)}")
print(f"  Games included ({MIN_GAMES}+ season games for both teams): {len(training_rows)}")
print(f"  Games dropped (insufficient season history): {skipped_insufficient_history}")
print(f"  Decay factor: {DECAY} (most recent game weight: {DECAY:.1f}/total)")


Processed 5000/20305 games, 4743 included...
Processed 10000/20305 games, 9488 included...
Processed 15000/20305 games, 14232 included...
Processed 20000/20305 games, 18980 included...

Processing complete:
  Total games processed: 20305
  Games included (4+ season games for both teams): 19285
  Games dropped (insufficient season history): 1020
  Decay factor: 0.9 (most recent game weight: 0.9/total)


In [13]:
# Convert to DataFrame and save
training_games3 = pd.DataFrame(training_rows)
print(f"Columns: {len(training_games3.columns)}")

Columns: 93


In [14]:
# save to training_games2.csv
training_games3.to_csv("data/training_games2.csv", index=False)

In [22]:
training_games3 = pd.read_csv("data/training_games2.csv")

In [23]:
def parse_roster_filename(path: str):
    fn = os.path.basename(path)
    m = re.match(r'^(?P<game_id>\d+)_(?P<away>[A-Z0-9]{2,4})_(?P<home>[A-Z0-9]{2,4})\.csv$', fn)
    if not m:
        raise ValueError(f"Bad roster filename format: {fn}")
    return m.group("game_id"), m.group("away"), m.group("home")


# Get all player stats files (game_rosters nested by year: data/game_rosters/2009/*.csv)
player_game_files = glob.glob("data/game_rosters/*/*.csv")
print(f"Found {len(player_game_files)} player game files to process")

# Track stats
games_processed = 0
players_updated = set()
games_without_date = []

# Process each game file
for game_file in player_game_files:
    # Extract game_id from filename (format: GAME_ID_AWAY_HOME.csv)
    try:
        filename, _, _ = parse_roster_filename(game_file)
        game_id = filename.zfill(10)  # schedule keys are zero-padded to 10 digi
        away_team, home_team = parse_roster_filename(game_file)[1:]
    except ValueError:
        print(f"Bad roster filename format: {game_file}")
        continue
    if away_team not in HIST_TO_MODERN or home_team not in HIST_TO_MODERN:
        continue

    # Look up game date
    if game_id not in game_date_map:
        games_without_date.append(game_id)
        continue
    
    game_date = game_date_map[game_id]
    
    # Load player stats for this game
    try:
        game_df = pd.read_csv(game_file, dtype={"personId": str, "PLAYER_ID": str})
    except Exception as e:
        print(f"Error reading {game_file}: {e}")
        continue
    
    if game_df.empty:
        continue
    
    # Determine player_id column name (could be personId or PLAYER_ID)
    player_id_col = None
    if "athlete_id" in game_df.columns:
        player_id_col = "athlete_id"
    elif "PLAYER_ID" in game_df.columns:
        player_id_col = "PLAYER_ID"
    else:
        print(f"Warning: No player ID column found in {game_file}")
        continue
    
    # Add game metadata to each row
    game_df["GAME_ID"] = game_id
    game_df["GAME_DATE"] = game_date
    
    # Process each player in this game
    for _, player_row in game_df.iterrows():
        player_id = str(player_row[player_id_col])
        
        if pd.isna(player_id) or player_id == "" or player_id == "nan":
            continue
        
        # Create/append to player's CSV file
        player_file = f"data/player_stats/{player_id}.csv"
        
        # Convert row to DataFrame for appending
        row_df = pd.DataFrame([player_row])
        
        if os.path.exists(player_file):
            # Append to existing file
            row_df.to_csv(player_file, mode='a', header=False, index=False)
        else:
            # Create new file with header
            row_df.to_csv(player_file, mode='w', header=True, index=False)
        
        players_updated.add(player_id)
    
    games_processed += 1
    
    # Progress indicator every 1000 games
    if games_processed % 1000 == 0:
        print(f"Processed {games_processed} games, {len(players_updated)} unique players so far...")

print(f"\n{'='*60}")
print(f"Summary:")
print(f"  Games processed: {games_processed}")
print(f"  Unique players: {len(players_updated)}")
print(f"  Games without date mapping: {len(games_without_date)}")

if games_without_date:
    print(f"\nSample games without dates: {games_without_date[:5]}")

Found 20377 player game files to process
Processed 1000 games, 504 unique players so far...
Processed 2000 games, 601 unique players so far...
Processed 3000 games, 1069 unique players so far...
Processed 4000 games, 1293 unique players so far...
Processed 5000 games, 1355 unique players so far...
Processed 6000 games, 1409 unique players so far...
Processed 7000 games, 1424 unique players so far...
Processed 8000 games, 1479 unique players so far...
Processed 9000 games, 1485 unique players so far...
Processed 10000 games, 1569 unique players so far...
Processed 11000 games, 1647 unique players so far...
Processed 12000 games, 1863 unique players so far...
Bad roster filename format: data/game_rosters/2026/401838140_WORLD_STARS.csv
Bad roster filename format: data/game_rosters/2026/401838141_STARS_STRIPES.csv
Bad roster filename format: data/game_rosters/2026/401838142_WORLD_STRIPES.csv
Bad roster filename format: data/game_rosters/2026/401838143_STARS_STRIPES.csv
Processed 13000 game

In [30]:
# Load nba_teams.csv to map team_id to abbreviation
# Canonicalize abbreviations so ESPN variants (GS, SA, NY, etc.) and
# historical names (SEA, NJ) all map to modern canonical form (GSW, SAS, NYK, OKC, BKN, etc.)
teams_df = pd.read_csv("data/nba_teams.csv", dtype={"team_id": str})
team_id_to_abbr = {tid: canonical(abbr) for tid, abbr in zip(teams_df["team_id"], teams_df["team_code"])}
print(f"Loaded {len(team_id_to_abbr)} team mappings")
print(f"Sample: { {k: team_id_to_abbr[k] for k in list(team_id_to_abbr)[:5]} }")

Loaded 30 team mappings
Sample: {'1': 'ATL', '10': 'HOU', '11': 'IND', '12': 'LAC', '13': 'LAL'}


In [31]:
import csv

def read_player_csv(path):
    """Tolerant reader for player_stats CSVs whose schema grew over time: newer
    rows carry extra trailing columns the header never named, which makes pandas'
    C parser abort the whole file. We read manually and truncate each row to the
    header width (every column we need lives in the stable leading prefix), so all
    rows are kept and no parse error is silently swallowed."""
    with open(path, newline="") as f:
        r = csv.reader(f)
        header = next(r)
        n = len(header)
        rows = [row[:n] for row in r if len(row) >= n]
    return pd.DataFrame(rows, columns=header)


# Helper: get a player's average minutes from their PRIOR games (not current game).
# Used for weighting so we don't leak current-game playing time.
_prior_min_cache = {}

def get_prior_avg_minutes(player_id, current_game_id, current_game_date, n=5):
    """Return player's average minutes over last n games BEFORE current_game_date."""
    cache_key = (player_id, str(current_game_id))
    if cache_key in _prior_min_cache:
        return _prior_min_cache[cache_key]
    try:
        pid_int = int(float(player_id))
    except (ValueError, TypeError):
        _prior_min_cache[cache_key] = 0.0
        return 0.0
    pfile = f"data/player_stats/{pid_int}.csv"
    if not os.path.exists(pfile):
        _prior_min_cache[cache_key] = 0.0
        return 0.0
    pdf = read_player_csv(pfile)
    gdt_col = "GAME_DATE" if "GAME_DATE" in pdf.columns else ("game_date" if "game_date" in pdf.columns else None)
    min_col = "MIN" if "MIN" in pdf.columns else ("minutes" if "minutes" in pdf.columns else None)
    if not gdt_col or not min_col:
        _prior_min_cache[cache_key] = 0.0
        return 0.0
    pdf[gdt_col] = pd.to_datetime(pdf[gdt_col], errors='coerce')
    pdf['_min'] = pdf[min_col].apply(convert_minutes_to_decimal)
    cur_dt = pd.to_datetime(current_game_date, errors='coerce')
    prior = pdf[(pdf[gdt_col] < cur_dt) & (pdf['_min'] > 0)]
    if prior.empty:
        _prior_min_cache[cache_key] = 0.0
        return 0.0
    avg = float(prior.sort_values(gdt_col, ascending=False).head(n)['_min'].mean())
    _prior_min_cache[cache_key] = avg
    return avg

# Process each game and add weighted player info by home/away team.
# Roster = all players listed in the game's roster file (known pre-game).
# Weights = prior average minutes (from past games, no leakage).
# Players with no prior history naturally get weight 0 and are excluded.

training_rows = []
games_with_players = 0
games_without_players = 0

for idx, game_row in training_games3.iterrows():
    game_id = str(game_row["game_id"])
    
    # Start with all game data
    training_row = game_row.to_dict()
    
    # Canonicalize team abbreviations so ESPN variants (GS, SA, NY, SEA, NJ, etc.)
    # match the canonical form used in team_id_to_abbr (GSW, SAS, NYK, OKC, BKN, etc.)
    home_team_abbr = canonical(game_row["home_team"])
    away_team_abbr = canonical(game_row["away_team"])
    
    # Look up player stats for this game
    if game_id not in game_id_to_file:
        games_without_players += 1
        training_rows.append(training_row)
        continue
    
    # Load player stats
    try:
        players_df = pd.read_csv(
            game_id_to_file[game_id], 
            dtype={"personId": str, "PLAYER_ID": str, "teamId": str, "TEAM_ID": str,
                   "athlete_id": str, "team_id": str}
        )
    except Exception as e:
        games_without_players += 1
        training_rows.append(training_row)
        continue
    
    if players_df.empty:
        games_without_players += 1
        training_rows.append(training_row)
        continue
    
    # Determine column names
    player_id_col = "athlete_id" if "athlete_id" in players_df.columns else ("PLAYER_ID" if "PLAYER_ID" in players_df.columns else None)
    team_id_col = "team_id" if "team_id" in players_df.columns else ("TEAM_ID" if "TEAM_ID" in players_df.columns else None)
    
    if not player_id_col or not team_id_col:
        games_without_players += 1
        training_rows.append(training_row)
        continue
    
    # Use roster players who actually played (filter out DNPs).
    # did_not_play status is known pre-game from injury reports / lineup submissions.
    roster_players = players_df.copy()
    dnp_col = None
    for c in ['did_not_play', 'didNotPlay']:
        if c in roster_players.columns:
            dnp_col = c
            break
    if dnp_col is not None:
        roster_players = roster_players[roster_players[dnp_col] != True].copy()
        roster_players = roster_players[roster_players[dnp_col].astype(str).str.lower() != 'true'].copy()
    
    # Get unique teams (should be 2)
    unique_teams = roster_players[team_id_col].dropna().unique()
    
    if len(unique_teams) < 2:
        games_without_players += 1
        training_rows.append(training_row)
        continue
    
    # Map team_ids to abbreviations to determine which is home/away
    home_team_id = None
    away_team_id = None
    
    for team_id in unique_teams:
        team_id_str = str(team_id)
        team_abbr = team_id_to_abbr.get(team_id_str, None)
        
        if team_abbr == home_team_abbr:
            home_team_id = team_id_str
        elif team_abbr == away_team_abbr:
            away_team_id = team_id_str
    
    # Skip if we can't match both teams
    if not home_team_id or not away_team_id:
        games_without_players += 1
        training_rows.append(training_row)
        continue
    
    cur_game_date = game_row.get("game_date", None)
    
    # Process away team players
    away_players = roster_players[roster_players[team_id_col].astype(str) == away_team_id].copy()

    if not away_players.empty:
        # Weight by PRIOR average minutes (not current-game minutes) to avoid leakage
        prior_mins = [get_prior_avg_minutes(str(p[player_id_col]), game_id, cur_game_date)
                      for _, p in away_players.iterrows()]
        away_players["prior_avg_min"] = prior_mins
        # Only keep players with nonzero prior history (rookies/new players excluded)
        away_players = away_players[away_players["prior_avg_min"] > 0].copy()
        total_prior = away_players["prior_avg_min"].sum()
        if total_prior > 0:
            away_players["weight"] = away_players["prior_avg_min"] / total_prior
        else:
            away_players["weight"] = 0.0

        for i, (_, player) in enumerate(away_players.iterrows(), 1):
            training_row[f"away_player_{i}_id"] = str(player[player_id_col])
            training_row[f"away_player_{i}_weight"] = float(player["weight"])

    # Process home team players
    home_players = roster_players[roster_players[team_id_col].astype(str) == home_team_id].copy()

    if not home_players.empty:
        prior_mins = [get_prior_avg_minutes(str(p[player_id_col]), game_id, cur_game_date)
                      for _, p in home_players.iterrows()]
        home_players["prior_avg_min"] = prior_mins
        home_players = home_players[home_players["prior_avg_min"] > 0].copy()
        total_prior = home_players["prior_avg_min"].sum()
        if total_prior > 0:
            home_players["weight"] = home_players["prior_avg_min"] / total_prior
        else:
            home_players["weight"] = 0.0

        for i, (_, player) in enumerate(home_players.iterrows(), 1):
            training_row[f"home_player_{i}_id"] = str(player[player_id_col])
            training_row[f"home_player_{i}_weight"] = float(player["weight"])
    
    training_rows.append(training_row)
    games_with_players += 1
    
    # Progress indicator
    if (idx + 1) % 5000 == 0:
        print(f"Processed {idx + 1}/{len(training_games3)} games, {games_with_players} with players...")

print(f"\n{'='*60}")
print(f"Processing complete:")
print(f"  Games with player data: {games_with_players}")
print(f"  Games without player data: {games_without_players}")
print(f"  Total: {len(training_rows)}")

Processed 5000/19285 games, 4999 with players...
Processed 10000/19285 games, 9998 with players...
Processed 15000/19285 games, 14998 with players...

Processing complete:
  Games with player data: 19283
  Games without player data: 2
  Total: 19285


In [32]:
training_games3 = pd.DataFrame(training_rows)
print(f"Columns: {len(training_games3.columns)}")

Columns: 161


In [33]:
import csv

def read_player_csv(path):
    """Tolerant reader for player_stats CSVs whose schema grew over time: newer
    rows carry extra trailing columns the header never named, which makes pandas'
    C parser abort the whole file. We read manually and truncate each row to the
    header width (every column we need lives in the stable leading prefix), so all
    rows are kept and no parse error is silently swallowed."""
    with open(path, newline="") as f:
        r = csv.reader(f)
        header = next(r)
        n = len(header)
        rows = [row[:n] for row in r if len(row) >= n]
    return pd.DataFrame(rows, columns=header)


# Stats to aggregate per player. Count stats are normalized to per-minute rates;
# percentage stats (FG_PCT, FG3_PCT, FT_PCT) are kept as-is.
PLAYER_COUNT_COLS = ['FGA', 'FG3M', 'FG3A', 'FTM', 'FTA',
                     'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS']
PLAYER_PCT_COLS = ['FG_PCT', 'FG3_PCT', 'FT_PCT']
PLAYER_STAT_COLS = PLAYER_COUNT_COLS + PLAYER_PCT_COLS

MIN_MINUTES_FOR_RATE = 5.0  # Only include games with >= 5 min for stable per-minute rates

# ESPN player_stats files use long column names; map them to short stat names
ESPN_TO_SHORT = {
    'field_goals_made': 'FGM',
    'field_goals_attempted': 'FGA',
    'three_point_field_goals_made': 'FG3M',
    'three_point_field_goals_attempted': 'FG3A',
    'free_throws_made': 'FTM',
    'free_throws_attempted': 'FTA',
    'offensive_rebounds': 'OREB',
    'defensive_rebounds': 'DREB',
    'rebounds': 'REB',
    'assists': 'AST',
    'steals': 'STL',
    'blocks': 'BLK',
    'turnovers': 'TO',
    'fouls': 'PF',
    'points': 'PTS',
    'plus_minus': 'PLUS_MINUS',
}

WINDOWS = [5, 10, 20]  # 5-game, 10-game, and 20-game rolling windows

# Use training_games3 (has player roster columns)
df_src = training_games3.copy()
print(f"Source games: {len(df_src)}")

# Always rebuild from scratch to avoid stale/corrupted data
output_file = "data/training_games3.csv"
print(f"Building {output_file} from scratch...")

# Cache for loaded player stats
player_stats_cache = {}

def load_player_stats(player_id):
    """Load player stats CSV, return DataFrame or None."""
    if player_id in player_stats_cache:
        return player_stats_cache[player_id]
    try:
        player_id_int = int(float(player_id))
    except (ValueError, TypeError):
        player_stats_cache[player_id] = None
        return None
    player_file = f"data/player_stats/{player_id_int}.csv"
    if not os.path.exists(player_file):
        player_stats_cache[player_id] = None
        return None
    try:
        df = read_player_csv(player_file)
        # Rename ESPN long column names to short stat names
        rename_map = {k: v for k, v in ESPN_TO_SHORT.items() if k in df.columns and v not in df.columns}
        if rename_map:
            df = df.rename(columns=rename_map)
        # Also rename minutes column for consistency
        if "minutes" in df.columns and "MIN" not in df.columns:
            df = df.rename(columns={"minutes": "MIN"})
        # Compute derived percentage stats per-game from raw values
        for made, att, pct in [('FGM', 'FGA', 'FG_PCT'), ('FG3M', 'FG3A', 'FG3_PCT'), ('FTM', 'FTA', 'FT_PCT')]:
            if made in df.columns and att in df.columns and pct not in df.columns:
                m = pd.to_numeric(df[made], errors='coerce').fillna(0)
                a = pd.to_numeric(df[att], errors='coerce').fillna(0)
                df[pct] = np.where(a > 0, m / a, 0.0)
        player_stats_cache[player_id] = df
        return df
    except Exception as e:
        raise RuntimeError(f"Failed to load player file {player_file}: {e}") from e

def convert_min_to_numeric(min_str):
    """Convert MIN from MM:SS or numeric to float minutes."""
    if pd.isna(min_str):
        return 0.0
    try:
        return float(min_str)
    except (ValueError, TypeError):
        pass
    try:
        min_str = str(min_str).strip()
        if ':' in min_str:
            parts = min_str.split(':')
            return float(parts[0]) + (float(parts[1]) / 60.0 if len(parts) > 1 else 0.0)
        return float(min_str)
    except:
        return 0.0

def get_player_history(player_id, current_game_id, current_game_date, max_window=20):
    """
    Get player's recent per-minute stats (excluding current game).
    Count stats (PTS, REB, AST, etc.) are divided by minutes played per game.
    Percentage stats (FG_PCT, FG3_PCT, FT_PCT) are kept as-is.
    Only games with >= MIN_MINUTES_FOR_RATE minutes are included.
    Returns dict: {5: {stat: mean, ...}, 10: {stat: mean, ...}, 20: {stat: mean, ...}} or None.
    """
    df_player = load_player_stats(player_id)
    if df_player is None or df_player.empty:
        return None
    
    game_id_col = "GAME_ID" if "GAME_ID" in df_player.columns else ("game_id" if "game_id" in df_player.columns else None)
    game_date_col = "GAME_DATE" if "GAME_DATE" in df_player.columns else ("game_date" if "game_date" in df_player.columns else None)
    if not game_id_col or not game_date_col:
        return None
    
    df_p = df_player.copy()
    
    # Convert minutes and filter to games with enough playing time for stable rates
    if "MIN" in df_p.columns:
        df_p['min_numeric'] = df_p['MIN'].apply(convert_min_to_numeric)
        df_p = df_p[df_p['min_numeric'] >= MIN_MINUTES_FOR_RATE]
    else:
        return None
    
    # Exclude current game (no leakage)
    df_p = df_p[df_p[game_id_col].astype(str) != str(current_game_id)]
    if df_p.empty:
        return None
    
    # Sort by date descending, take up to max_window recent games
    df_p[game_date_col] = pd.to_datetime(df_p[game_date_col], errors='coerce')
    df_p = df_p.dropna(subset=[game_date_col])
    # Only include games BEFORE current game date
    current_dt = pd.to_datetime(current_game_date, errors='coerce')
    if current_dt is not None and not pd.isna(current_dt):
        df_p = df_p[df_p[game_date_col] < current_dt]
    df_p = df_p.sort_values(game_date_col, ascending=False).head(max_window)
    if df_p.empty:
        return None
    
    # Compute per-minute rates for count stats, keep pct stats as-is
    result = {}
    for w in WINDOWS:
        window_df = df_p.head(w)
        mins = window_df['min_numeric']
        stats = {}
        for stat in PLAYER_COUNT_COLS:
            if stat in window_df.columns:
                vals = pd.to_numeric(window_df[stat], errors='coerce').fillna(0)
                per_min = vals / mins
                stats[stat] = per_min.mean() if len(per_min) > 0 else 0.0
            else:
                stats[stat] = 0.0
        for stat in PLAYER_PCT_COLS:
            if stat in window_df.columns:
                vals = pd.to_numeric(window_df[stat], errors='coerce').dropna()
                stats[stat] = vals.mean() if len(vals) > 0 else 0.0
            else:
                stats[stat] = 0.0
        result[w] = stats
    
    return result

print("\nProcessing games: computing 5-game, 10-game, and 20-game weighted per-minute player averages...\n")

output_rows = []
total_processed = 0
dropped_no_data = 0

for idx, row in df_src.iterrows():
    game_id = str(row['game_id'])
    game_date = row['game_date']
    
    # Copy all non-player-roster columns
    new_row = {}
    for col in df_src.columns:
        if not (col.startswith('away_player_') or col.startswith('home_player_')):
            new_row[col] = row[col]
    
    # Initialize aggregated stats for each window: {window: {stat: 0.0}}
    away_agg = {w: {stat: 0.0 for stat in PLAYER_STAT_COLS} for w in WINDOWS}
    home_agg = {w: {stat: 0.0 for stat in PLAYER_STAT_COLS} for w in WINDOWS}
    
    has_away_data = False
    has_home_data = False
    
    # Process away team players
    for i in range(1, 16):
        pid_col = f'away_player_{i}_id'
        wt_col = f'away_player_{i}_weight'
        if pid_col not in row.index or pd.isna(row[pid_col]):
            continue
        player_id = row[pid_col]
        weight = row[wt_col] if wt_col in row.index and not pd.isna(row[wt_col]) else 0.0
        if weight == 0.0:
            continue
        
        history = get_player_history(player_id, game_id, game_date, max_window=20)
        if history is None:
            continue  # no prior data for this player, skip their contribution
        
        has_away_data = True
        for w in WINDOWS:
            for stat in PLAYER_STAT_COLS:
                away_agg[w][stat] += weight * history[w][stat]
    
    # Process home team players
    for i in range(1, 16):
        pid_col = f'home_player_{i}_id'
        wt_col = f'home_player_{i}_weight'
        if pid_col not in row.index or pd.isna(row[pid_col]):
            continue
        player_id = row[pid_col]
        weight = row[wt_col] if wt_col in row.index and not pd.isna(row[wt_col]) else 0.0
        if weight == 0.0:
            continue
        
        history = get_player_history(player_id, game_id, game_date, max_window=20)
        if history is None:
            continue
        
        has_home_data = True
        for w in WINDOWS:
            for stat in PLAYER_STAT_COLS:
                home_agg[w][stat] += weight * history[w][stat]
    
    # Drop the game if either team is missing player history.
    if not has_away_data or not has_home_data:
        dropped_no_data += 1
        total_processed += 1
        if total_processed % 1000 == 0:
            print(f"Processed {total_processed}/{len(df_src)} games...")
        continue
    
    # Add aggregated player stats for each window
    for w in WINDOWS:
        for stat in PLAYER_STAT_COLS:
            new_row[f'away_{w}g_player_{stat}'] = away_agg[w][stat]
            new_row[f'home_{w}g_player_{stat}'] = home_agg[w][stat]
    
    output_rows.append(new_row)
    total_processed += 1
    
    if total_processed % 1000 == 0:
        print(f"Processed {total_processed}/{len(df_src)} games, {dropped_no_data} dropped...")

# Write all at once
rolling_df = pd.DataFrame(output_rows)
rolling_df.to_csv(output_file, index=False)

print(f"\n{'='*60}")
print(f"Processing complete!")
print(f"  Total games processed: {total_processed}")
print(f"  Games kept: {total_processed - dropped_no_data}")
print(f"  Games dropped (no player history): {dropped_no_data}")
print(f"\nSaved to {output_file}")

Source games: 19285
Building data/training_games3.csv from scratch...

Processing games: computing 5-game, 10-game, and 20-game weighted per-minute player averages...

Processed 1000/19285 games, 1 dropped...
Processed 2000/19285 games, 1 dropped...
Processed 3000/19285 games, 1 dropped...
Processed 4000/19285 games, 1 dropped...
Processed 5000/19285 games, 1 dropped...
Processed 6000/19285 games, 1 dropped...
Processed 7000/19285 games, 1 dropped...
Processed 8000/19285 games, 1 dropped...
Processed 9000/19285 games, 1 dropped...
Processed 10000/19285 games, 2 dropped...
Processed 11000/19285 games, 2 dropped...
Processed 12000/19285 games, 2 dropped...
Processed 13000/19285 games, 2 dropped...
Processed 14000/19285 games, 2 dropped...
Processed 15000/19285 games, 2 dropped...
Processed 16000/19285 games, 2 dropped...
Processed 17000/19285 games, 2 dropped...
Processed 18000/19285 games, 2 dropped...
Processed 19000/19285 games, 2 dropped...

Processing complete!
  Total games process

In [34]:
# Verify training_games3.csv structure
print("Verifying training_games3.csv...")
df_verify = pd.read_csv("data/training_games3.csv", dtype={"game_id": str})

print(f"\\nShape: {df_verify.shape}")
print(f"Total columns: {len(df_verify.columns)}")

# Check for player roster columns (should be gone)
roster_cols = [col for col in df_verify.columns if ('_player_' in col and ('_id' in col or '_weight' in col))]
print(f"\\nRoster columns (should be 0): {len(roster_cols)}")
if roster_cols:
    print(f"  Found: {roster_cols[:5]}")

# Check for new aggregated player stat columns
away_player_stat_cols = [col for col in df_verify.columns if col.startswith('away_player_') and not col.endswith('_id') and not col.endswith('_weight')]
home_player_stat_cols = [col for col in df_verify.columns if col.startswith('home_player_') and not col.endswith('_id') and not col.endswith('_weight')]

print(f"\\nAway player stat columns: {len(away_player_stat_cols)}")
print(f"  {away_player_stat_cols}")

print(f"\\nHome player stat columns: {len(home_player_stat_cols)}")
print(f"  {home_player_stat_cols}")

# Check for preserved columns
print(f"\\naway_PTS_actual present: {'away_PTS_actual' in df_verify.columns}")
print(f"home_PTS_actual present: {'home_PTS_actual' in df_verify.columns}")

# Show sample row
print(f"\\nSample game:")
sample = df_verify.iloc[0]
print(f"Game ID: {sample['game_id']}")
print(f"Date: {sample['game_date']}")
print(f"Teams: {sample['away_team']} @ {sample['home_team']}")
print(f"\\nAway player stats (sample):")
for stat in ['FGA', 'FG_PCT', 'PTS', 'AST', 'REB']:
    col = f'away_player_{stat}'
    if col in sample:
        print(f"  {col}: {sample[col]:.4f}")
print(f"\\nHome player stats (sample):")
for stat in ['FGA', 'FG_PCT', 'PTS', 'AST', 'REB']:
    col = f'home_player_{stat}'
    if col in sample:
        print(f"  {col}: {sample[col]:.4f}")

print(f"\\nActual points:")
print(f"  Away: {sample['away_PTS_actual']}")
print(f"  Home: {sample['home_PTS_actual']}")

Verifying training_games3.csv...
\nShape: (19283, 201)
Total columns: 201
\nRoster columns (should be 0): 0
\nAway player stat columns: 0
  []
\nHome player stat columns: 0
  []
\naway_PTS_actual present: True
home_PTS_actual present: True
\nSample game:
Game ID: 301103001
Date: 2010-11-03
Teams: DET @ ATL
\nAway player stats (sample):
\nHome player stats (sample):
\nActual points:
  Away: 85
  Home: 94


In [35]:
nba_training_df = pd.read_csv('data/training_games3.csv')

In [36]:
nba_training_df = nba_training_df[nba_training_df['season'] >= '2008-09']
nba_training_df.drop(columns=['season', 'home_team_full', 'away_team_full'], inplace=True)
# Keep home_TEAM_CITY and away_TEAM_CITY for now — used to compute timezone difference downstream
nba_training_df.head()

,game_id,game_date,home_team,away_team,away_TEAM_CITY,away_FGM,away_FGA,away_FG_PCT,away_FG3M,away_FG3A,...,away_20g_player_PTS,home_20g_player_PTS,away_20g_player_PLUS_MINUS,home_20g_player_PLUS_MINUS,away_20g_player_FG_PCT,home_20g_player_FG_PCT,away_20g_player_FG3_PCT,home_20g_player_FG3_PCT,away_20g_player_FT_PCT,home_20g_player_FT_PCT
0,301103001,2010-11-03,ATL,DET,Detroit,35.554812,81.755452,0.434965,5.470195,14.181739,...,0.403237,0.438499,-0.177533,0.193063,0.433204,0.502098,0.227316,0.281488,0.517630,0.601346
1,301103002,2010-11-03,BOS,MIL,Milwaukee,30.522536,76.331201,0.401861,6.153533,19.705438,...,0.353066,0.382784,-0.118910,0.130960,0.414465,0.489237,0.175360,0.210383,0.485691,0.542373
2,301103023,2010-11-03,SAC,LAL,Los Angeles,42.023844,90.181448,0.467209,10.375400,22.471358,...,0.474301,0.419204,0.226406,-0.070936,0.453251,0.430257,0.332235,0.181811,0.512416,0.549733
3,301104022,2010-11-04,POR,OKC,Oklahoma City,32.817098,83.661530,0.392896,3.790637,20.143065,...,0.408089,0.404311,-0.129152,0.019352,0.386470,0.421446,0.126199,0.229196,0.611053,0.524345
4,301105011,2010-11-05,IND,MIL,Milwaukee,31.860101,78.692266,0.406046,5.871847,18.068008,...,0.351407,0.421924,-0.117320,-0.120446,0.401566,0.438863,0.172862,0.243503,0.520383,0.535048


In [37]:
nba_training_df.drop(columns=['away_PLUS_MINUS', 'home_PLUS_MINUS'], inplace=True)

In [38]:
categorical_variables = nba_training_df.select_dtypes(include=['number']).columns
print(categorical_variables)

Index(['game_id', 'away_FGM', 'away_FGA', 'away_FG_PCT', 'away_FG3M',
       'away_FG3A', 'away_FG3_PCT', 'away_FTM', 'away_FTA', 'away_FT_PCT',
       ...
       'away_20g_player_PTS', 'home_20g_player_PTS',
       'away_20g_player_PLUS_MINUS', 'home_20g_player_PLUS_MINUS',
       'away_20g_player_FG_PCT', 'home_20g_player_FG_PCT',
       'away_20g_player_FG3_PCT', 'home_20g_player_FG3_PCT',
       'away_20g_player_FT_PCT', 'home_20g_player_FT_PCT'],
      dtype='str', length=191)


In [39]:
nba_training_df['away_eFG'] = (nba_training_df['away_FGM'] + 0.5 * nba_training_df['away_FG3M']) / nba_training_df['away_FGA'] * 100
nba_training_df['home_eFG'] = (nba_training_df['home_FGM'] + 0.5 * nba_training_df['home_FG3M']) / nba_training_df['home_FGA'] * 100

/var/folders/qd/69pzjl_56vjdn7_xy8vbzwt00000gn/T/ipykernel_38935/1961225195.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  nba_training_df['away_eFG'] = (nba_training_df['away_FGM'] + 0.5 * nba_training_df['away_FG3M']) / nba_training_df['away_FGA'] * 100
/var/folders/qd/69pzjl_56vjdn7_xy8vbzwt00000gn/T/ipykernel_38935/1961225195.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  nba_training_df['home_eFG'] = (nba_training_df['home_FGM'] + 0.5 * nba_training_df['home_FG3M']) / nba_training_df['home_FGA'] * 100


In [40]:
nba_training_df['home_win_pct'] = nba_training_df['home_wins'] / (nba_training_df['home_wins'] + nba_training_df['home_losses'] + 0.01)

/var/folders/qd/69pzjl_56vjdn7_xy8vbzwt00000gn/T/ipykernel_38935/566053995.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  nba_training_df['home_win_pct'] = nba_training_df['home_wins'] / (nba_training_df['home_wins'] + nba_training_df['home_losses'] + 0.01)


In [41]:
nba_training_df['away_win_pct'] = nba_training_df['away_wins'] / (nba_training_df['away_wins'] + nba_training_df['away_losses'] + 0.01)

/var/folders/qd/69pzjl_56vjdn7_xy8vbzwt00000gn/T/ipykernel_38935/2210808479.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  nba_training_df['away_win_pct'] = nba_training_df['away_wins'] / (nba_training_df['away_wins'] + nba_training_df['away_losses'] + 0.01)


In [42]:
nba_training_df.head()

,game_id,game_date,home_team,away_team,away_TEAM_CITY,away_FGM,away_FGA,away_FG_PCT,away_FG3M,away_FG3A,...,away_20g_player_FG_PCT,home_20g_player_FG_PCT,away_20g_player_FG3_PCT,home_20g_player_FG3_PCT,away_20g_player_FT_PCT,home_20g_player_FT_PCT,away_eFG,home_eFG,home_win_pct,away_win_pct
0,301103001,2010-11-03,ATL,DET,Detroit,35.554812,81.755452,0.434965,5.470195,14.181739,...,0.433204,0.502098,0.227316,0.281488,0.517630,0.601346,46.834687,50.556781,0.997506,0.000000
1,301103002,2010-11-03,BOS,MIL,Milwaukee,30.522536,76.331201,0.401861,6.153533,19.705438,...,0.414465,0.489237,0.175360,0.210383,0.485691,0.542373,44.017783,52.426516,0.748130,0.249377
2,301103023,2010-11-03,SAC,LAL,Los Angeles,42.023844,90.181448,0.467209,10.375400,22.471358,...,0.453251,0.430257,0.332235,0.181811,0.512416,0.549733,52.351725,50.256793,0.748130,0.997506
3,301104022,2010-11-04,POR,OKC,Oklahoma City,32.817098,83.661530,0.392896,3.790637,20.143065,...,0.386470,0.421446,0.126199,0.229196,0.611053,0.524345,41.491491,48.490875,0.798403,0.498753
4,301105011,2010-11-05,IND,MIL,Milwaukee,31.860101,78.692266,0.406046,5.871847,18.068008,...,0.401566,0.438863,0.172862,0.243503,0.520383,0.535048,44.217845,46.468850,0.498753,0.199601


In [43]:
# Encode team identity as integer IDs (alphabetical order, 0-29)
# Tree models can learn team-specific effects (dynasty teams, perennial losers, etc.)
# Must canonicalize first since team_stats uses ESPN-style codes (GS, SA, NY, etc.)
TEAM_TO_ID = {abbr: i for i, abbr in enumerate(sorted([
    'ATL', 'BOS', 'BKN', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW',
    'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK',
    'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS'
]))}

nba_training_df['home_team_id'] = nba_training_df['home_team'].apply(canonical).map(TEAM_TO_ID)
nba_training_df['away_team_id'] = nba_training_df['away_team'].apply(canonical).map(TEAM_TO_ID)

unmapped = nba_training_df[nba_training_df['home_team_id'].isna() | nba_training_df['away_team_id'].isna()]
if len(unmapped) > 0:
    print(f"WARNING: {len(unmapped)} rows with unmapped teams:")
    print(unmapped[['home_team', 'away_team']].drop_duplicates())
else:
    print(f"All teams mapped to IDs 0-29.")

# ── Timezone difference: home_tz - away_tz (hours) ──
# Positive = away team traveled east, negative = away team traveled west
# UTC offsets: Eastern=-5, Central=-6, Mountain=-7, Pacific=-8
CITY_TZ = {
    # Eastern (-5)
    'Atlanta': -5, 'Boston': -5, 'Brooklyn': -5, 'Charlotte': -5,
    'Cleveland': -5, 'Detroit': -5, 'Indiana': -5, 'Miami': -5,
    'New Jersey': -5, 'New York': -5, 'Orlando': -5, 'Philadelphia': -5,
    'Toronto': -5, 'Washington': -5,
    # Central (-6)
    'Chicago': -6, 'Dallas': -6, 'Houston': -6, 'Memphis': -6,
    'Milwaukee': -6, 'Minnesota': -6, 'New Orleans': -6,
    'NO/Oklahoma City': -6, 'Oklahoma City': -6, 'San Antonio': -6,
    # Mountain (-7)
    'Denver': -7, 'Utah': -7, 'Phoenix': -7,
    # Pacific (-8)
    'Golden State': -8, 'LA': -8, 'Los Angeles': -8,
    'Portland': -8, 'Sacramento': -8, 'Seattle': -8,
}

# Clean city names (strip whitespace / \r\n) then map to UTC offset
nba_training_df['_home_city'] = nba_training_df['home_TEAM_CITY'].str.strip()
nba_training_df['_away_city'] = nba_training_df['away_TEAM_CITY'].str.strip()
nba_training_df['_home_tz'] = nba_training_df['_home_city'].map(CITY_TZ)
nba_training_df['_away_tz'] = nba_training_df['_away_city'].map(CITY_TZ)

# tz_diff: home_tz - away_tz
# e.g. LAL(home,-8) vs NYK(away,-5) → -8 - (-5) = -3 (away team traveled west)
# e.g. MIA(home,-5) vs GSW(away,-8) → -5 - (-8) = +3 (away team traveled east)
nba_training_df['tz_diff'] = nba_training_df['_home_tz'] - nba_training_df['_away_tz']

# Check for unmapped cities
tz_unmapped = nba_training_df[nba_training_df['tz_diff'].isna()]
if len(tz_unmapped) > 0:
    print(f"\nWARNING: {len(tz_unmapped)} rows with unmapped city timezone:")
    print(tz_unmapped[['_home_city', '_away_city']].drop_duplicates())
else:
    print(f"\nTimezone diff computed for all rows.")

print(f"\ntz_diff distribution:")
print(nba_training_df['tz_diff'].value_counts().sort_index())

# Drop helper columns and original city columns
nba_training_df.drop(columns=['_home_city', '_away_city', '_home_tz', '_away_tz',
                               'home_TEAM_CITY', 'away_TEAM_CITY'], inplace=True)

All teams mapped to IDs 0-29.

Timezone diff computed for all rows.

tz_diff distribution:
tz_diff
-3    1001
-2    1778
-1    3236
 0    7228
 1    3247
 2    1800
 3     993
Name: count, dtype: int64


/var/folders/qd/69pzjl_56vjdn7_xy8vbzwt00000gn/T/ipykernel_38935/2680655155.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  nba_training_df['home_team_id'] = nba_training_df['home_team'].apply(canonical).map(TEAM_TO_ID)
/var/folders/qd/69pzjl_56vjdn7_xy8vbzwt00000gn/T/ipykernel_38935/2680655155.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  nba_training_df['away_team_id'] = nba_training_df['away_team'].apply(canonical).map(TEAM_TO_ID)
/var/folders/qd/69pzjl_56vjdn7_xy8vbzwt00000gn/T/ipykernel_38935/2680655155.py:41: P

In [44]:
# -- Compute r10 eFG (rolling 10-game effective field goal %) --
nba_training_df['away_r10_eFG'] = (nba_training_df['away_r10_FGM'] + 0.5 * nba_training_df['away_r10_FG3M']) / nba_training_df['away_r10_FGA'].clip(lower=1) * 100
nba_training_df['home_r10_eFG'] = (nba_training_df['home_r10_FGM'] + 0.5 * nba_training_df['home_r10_FG3M']) / nba_training_df['home_r10_FGA'].clip(lower=1) * 100

# -- Drop fully-NA columns --
nba_training_df.drop(columns=['home_r10_PLUS_MINUS', 'away_r10_PLUS_MINUS'], inplace=True)

# -- Drop rows where r10 fields are NA (early-season games with <10 team games played) --
r10_cols = [c for c in nba_training_df.columns if 'r10' in c]
before = len(nba_training_df)
nba_training_df.dropna(subset=r10_cols, inplace=True)
print(f'Dropped {before - len(nba_training_df)} rows with NA r10 fields (early-season cold start)')

nba_training_df['score_diff'] = nba_training_df['home_PTS_actual'] - nba_training_df['away_PTS_actual']
nba_training_df.drop(columns=['home_PTS_actual', 'away_PTS_actual'], inplace=True)
nba_training_df.head()

nba_training_df.to_csv('data/training_games4.csv', index=False)
print(f'Saved {len(nba_training_df)} rows to data/training_games4.csv')

/var/folders/qd/69pzjl_56vjdn7_xy8vbzwt00000gn/T/ipykernel_38935/3998019502.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  nba_training_df['away_r10_eFG'] = (nba_training_df['away_r10_FGM'] + 0.5 * nba_training_df['away_r10_FG3M']) / nba_training_df['away_r10_FGA'].clip(lower=1) * 100
/var/folders/qd/69pzjl_56vjdn7_xy8vbzwt00000gn/T/ipykernel_38935/3998019502.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  nba_training_df['home_r10_eFG'] = (nba_training_df['home_r10_FGM'] + 0.5 * nba_training_df['home_r10_FG3M']) / nba_

Dropped 1477 rows with NA r10 fields (early-season cold start)
Saved 17806 rows to data/training_games4.csv
